# Understanding counting: accuracy and inference time

Test the **same 40 images** with the existing trained scorers:

| Mode | Scorer | Image-token FFN computation |
|---|---|---|
| Dense | Unneeded | 100% |
| Shared | $c_l(h)$ (`f`) | 90%, rounded up |
| Specialized | $c_l(h)+u_l(h)$ | 90%, rounded up |

Routing applies during prefill at layers 0–26. Attention, text tokens, layer 27,
and answer decoding stay dense. This notebook trains nothing.

## 1. Install, mount Drive, and clone the repository
Open this notebook from GitHub in a **fresh Colab GPU runtime**.
The FFN and inference implementation is in `counting_experiment/understanding_eval.py`.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.46.3", "tokenizers==0.20.3", "diffusers==0.30.1",
                "accelerate", "einops", "omegaconf", "timm", "decord",
                "Pillow", "numpy", "pandas", "matplotlib"], check=True)

from google.colab import drive
drive.mount("/content/drive")

REPO_ROOT = Path("/content/FlashU-Lab")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch",
                    "feature/pilot-acceleration-experiment",
                    "https://github.com/Nicerova7/FlashU-Lab.git", str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import json
import torch
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from PIL import Image
from torchvision import transforms
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download
from IPython.display import display
from counting_experiment.understanding_eval import (
    load_test_cases, load_scorers, read_checkpoint_u_records, check_held_out,
    run_evaluation, summarize_evaluation, sha256_file,
)
from evaluation.calculate_token_importance_understanding import load_model_resources

## 2. Load and preview the 40 test images
The JPEGs live directly in `MyDrive/intership/counting_test_images/`.
Questions and reference counts come from the cloned repository.

Check the grid: each **center crop** must still show the reference number of
the queried object. Resolve a mismatch before testing any mode.

In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/intership")
TEST_DIR = REPO_ROOT / "counting_experiment/test"
IMAGE_DIR = DATA_ROOT / "counting_test_images"
WEIGHTS = DATA_ROOT / "gradient_mlp_results/depth_router_minimal_v2/routers.pt"
U_ROOT = DATA_ROOT / "gradient_update_understanding"
VAE_PATH = DATA_ROOT / "model_weights/Wan2.1_VAE.pth"
RESULTS_DIR = DATA_ROOT / "counting_experiment/results" / datetime.now(timezone.utc).strftime("understanding_%Y%m%dT%H%M%S_%fZ")

ROUTER_SEED = 42
EVAL_SEED = 1001
KEEP_FRACTION = 0.90
MAX_NEW_TOKENS = 16
TIMING_REPEATS = 3

cases = load_test_cases(TEST_DIR, image_dir=IMAGE_DIR)
config = OmegaConf.load(REPO_ROOT / "configs/showo2_1.5b_demo_432x432.yaml")
resolution = int(config.dataset.preprocessing.resolution)
preview = transforms.Compose([
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop((resolution, resolution)),
])
fig, axes = plt.subplots(8, 5, figsize=(15, 24))
for ax, case in zip(axes.flat, cases):
    with Image.open(case["resolved_image"]) as image:
        ax.imshow(preview(image.convert("RGB")))
    ax.set_title(f"{case['id']}: {case['target_count']} {case['object_class']}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Load the trained scorers and Show-o2
Load $c$ and $u$ from the existing v2 checkpoint. Check the test images against
its original U collection once, then load the frozen model at collection precision.

In [ ]:
scorers, saved = load_scorers(WEIGHTS, ROUTER_SEED)
used_records, feature_dim, dtype_name = read_checkpoint_u_records(saved, U_ROOT)
holdout = check_held_out(cases, used_records)
print("Test images separate from previous U collection:", holdout["overlap_images"] == 0)
del saved

if not VAE_PATH.exists():
    hf_hub_download(repo_id="Wan-AI/Wan2.1-T2V-14B", filename=VAE_PATH.name,
                    local_dir=VAE_PATH.parent)
config.model.vae_model.pretrained_model_path = str(VAE_PATH)
config.layers = "all"
config.vae_deterministic = False
resources = load_model_resources(config, torch.device("cuda"), getattr(torch, dtype_name))
assert len(resources.model.showo.model.layers) == 28
assert resources.model.showo.model.config.hidden_size == feature_dim
print("Loaded:", config.model.showo.pretrained_model_path, dtype_name)

## 4. Run dense, shared, and specialized inference
Warm up each mode once. Run each image in all three modes with identical seeds,
repeating three times and rotating mode order for timing.

Time includes preprocessing, VAE encoding, prompt assembly, FFN routing,
prefill and answer decoding. It excludes model loading, JPEG file reads,
result saving, reference scoring and warm-up.

In [ ]:
rows = run_evaluation(cases, resources, config, scorers, RESULTS_DIR,
                      seed=EVAL_SEED, keep_fraction=KEEP_FRACTION,
                      max_new_tokens=MAX_NEW_TOKENS, repeats=TIMING_REPEATS)

## 5. Compare accuracy and time; save the results
Accuracy is correct counts / 40 images; unparseable answers count as wrong.
Latency uses the median repeat per image, then averages over images.
Speedup is dense latency / routed latency: **greater than 1 means faster**.

Specialized versus shared tests whether $u$ helps at the same skip budget.
Either routed mode versus dense tests the quality/speed trade-off.

In [ ]:
per_image, summary = summarize_evaluation(rows)
per_image.to_csv(RESULTS_DIR / "per_image.csv", index=False)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)

settings = dict(checkpoint=str(WEIGHTS), checkpoint_sha256=sha256_file(WEIGHTS),
                router_seed=ROUTER_SEED, eval_seed=EVAL_SEED,
                image_directory=str(IMAGE_DIR),
                test_manifest_sha256=sha256_file(TEST_DIR / "understanding.json"),
                keep_fraction=KEEP_FRACTION, max_new_tokens=MAX_NEW_TOKENS,
                timing_repeats=TIMING_REPEATS, holdout=holdout,
                model_config=OmegaConf.to_container(config, resolve=True),
                precision=dtype_name, gpu=torch.cuda.get_device_name())
(RESULTS_DIR / "run.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
display(summary)
print("Saved to:", RESULTS_DIR)